# football-laya — eval n_2000 (calibracao + teste)

Usa os pesos do n_2000 vindos do output da run de treino (kernel source). Avalia base (0) e n_2000 no calib+test fixos.


In [ ]:
!nvidia-smi -L
import torch
print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
!pip install -q laya==0.3.26

In [ ]:
import shutil, zipfile
from pathlib import Path

roots = [Path('/kaggle/input')]
zips = list(Path('/kaggle/input').rglob('*.zip'))
if zips:  # dataset subido com --dir-mode zip: extrair primeiro
    tmp = Path('/tmp/ds')
    for z in zips:
        with zipfile.ZipFile(z) as f:
            f.extractall(tmp / z.stem)
    roots.append(tmp)

def find_dir(marker):
    for r in roots:
        hits = list(r.rglob(marker))
        if hits:
            return hits[0].parent
    raise AssertionError(marker + ' nao encontrado: liga o dataset em Add input!')

DST = Path('/kaggle/working/fb')
for marker, dest in (('processed/train.jsonl', DST / 'data' / 'processed'),
                     ('rl_agent_config.json', DST / 'data' / 'models' / 'laya-multilingual-base'),
                     ('scripts/train_curve.py', DST / 'scripts')):
    src = find_dir(marker)
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(src, dest)
print('tree pronta')


In [ ]:
# traz o n_2000 para a arvore de trabalho (robusto a zips e a nomes)
import shutil, zipfile
from pathlib import Path
roots = [Path('/kaggle/input')]
if Path('/tmp/ds').exists():
    roots.append(Path('/tmp/ds'))
for z in list(Path('/kaggle/input').rglob('*.zip')):
    dest = Path('/tmp/ds') / z.stem
    if not dest.exists():
        with zipfile.ZipFile(z) as f:
            f.extractall(dest)
        roots.append(Path('/tmp/ds'))
cands = []
for r in roots:
    cands += [p for p in r.rglob('model.safetensors') if 'laya-multilingual-base' not in str(p)]
print('candidatos pesos:', [str(p) for p in cands], flush=True)
assert cands, 'pesos n2000 nao encontrados no input: verifica Add input dos datasets!'
pref = [p for p in cands if 'n_2000' in str(p)] or [p for p in cands if 'n2000' in str(p).lower()] or cands
src = pref[0].parent
print('a usar:', str(pref[0]), flush=True)
dest = Path('/kaggle/working/fb/data/models/curve_kaggle/n_2000')
dest.mkdir(parents=True, exist_ok=True)
for f in src.iterdir():
    d = dest / f.name
    if d.exists():
        shutil.rmtree(d) if d.is_dir() else d.unlink()
    shutil.copytree(f, d) if f.is_dir() else shutil.copy2(f, d)
print('n_2000 pronto:', sorted(p.name for p in dest.iterdir()))


In [ ]:
# forward so-teste em streaming (nao precisa de re-subir o dataset).
# batch 16 + AMP: 128x1024 rebenta a T4 16GB.
import json, time
from pathlib import Path
import torch
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya.common import QTYPES, build_model, build_sequence, collate_items
from safetensors.torch import load_file

N, BATCH, CHUNK, MAX_CASES = 2000, 16, 50, 0  # MAX_CASES=20 para smoke test
MDIR = Path('/kaggle/working/fb/data/models/curve_kaggle') / f'n_{N}'
TDIR = MDIR  # n_2000 traz tokenizer+encoder proprios
rows = [json.loads(l) for l in open('/kaggle/working/fb/data/processed/test.jsonl') if l.strip()]
rows.sort(key=lambda r: r['kickoff_utc'])
if MAX_CASES:
    rows = rows[:MAX_CASES]
_fix_tokenizer_config(str(MDIR))
cfg = json.load(open(MDIR / 'rl_agent_config.json'))
tok = AutoTokenizer.from_pretrained(TDIR / 'tokenizer')
model = build_model(cfg, encoder_dir=str(MDIR / 'encoder'), pretrained=False)
model.load_state_dict(load_file(str(MDIR / 'model.safetensors')), strict=True)
dev = torch.device('cuda')
model.to(dev).eval()
print(f'modelo n_{N} OK, casos={len(rows)}', flush=True)
tot, skip, nb, t0 = 0, 0, 0, time.time()
with torch.no_grad():
    for c0 in range(0, len(rows), CHUNK):
        flat = []
        for row in rows[c0:c0 + CHUNK]:
            for qid, q in row['questions'].items():
                if qid not in row['labels']:
                    skip += 1
                    continue
                keys = ['false', 'true'] if q['type'] == 'noul' else list(q.get('criteria', {}).keys())
                seq, mk = build_sequence(tok, row['state_text'], {'t': q['type'], 'ins': q['instructions'], 'crit': q.get('criteria', {})}, cfg.get('max_len', 1024), cfg.get('head_max_len', 256))
                if len(mk) != len(keys):
                    skip += 1
                    continue
                flat.append({'ids': seq, 'markers': mk, 'qtype': QTYPES[q['type']]})
        bs = BATCH
        bi = 0
        while bi < len(flat):
            try:
                b = collate_items([flat[bi:bi + bs]], tok.pad_token_id)
                with torch.amp.autocast('cuda', dtype=torch.float16):
                    lg, _ = model(b['input_ids'].to(dev), b['attention_mask'].to(dev), b['marker_pos'].to(dev), b['marker_mask'].to(dev), b['qtype'].to(dev))
                del lg
                nb += 1
                bi += bs
            except RuntimeError as e:
                if 'out of memory' not in str(e).lower() or bs <= 1:
                    raise
                torch.cuda.empty_cache()
                bs = max(1, bs // 2)
                print(f'OOM -> batch {bs}', flush=True)
        tot += len(flat)
        print(f'casos {min(c0 + CHUNK, len(rows))}/{len(rows)} items={tot} batches={nb} {time.time() - t0:.0f}s mem={torch.cuda.memory_allocated() / 1e9:.2f}GB', flush=True)
print(f'DONE n_{N}: items={tot} skip={skip} batches={nb} {time.time() - t0:.0f}s', flush=True)


In [ ]:
!python /kaggle/working/fb/scripts/eval_curve.py --checkpoints 0,2000 --batch 16 --curve-dir /kaggle/working/fb/data/models/curve_kaggle --metrics-dir /kaggle/working/fb/data/processed/curve_kaggle


In [ ]:
!cat /kaggle/working/fb/data/processed/curve_kaggle/metrics.csv


Saca o  no fim (e  para a reliability dos nouls).
